# 00 — Protocol and Data Audit

This notebook verifies the **frozen data and evaluation contract** before any model is tuned or any test result is inspected.

The four datasets are intentionally reduced to the same model-ready schema. Every derived input is rebuilt with one shared function so that differences in forecasting performance cannot be attributed to dataset-specific feature engineering code.

### Primary information boundary

- Historical demand and weather may be used only when their source time is no later than the forecast origin.
- Future calendar variables are deterministic and therefore available.
- Realized weather at target hours `t+1 ... t+24` is **not** supplied to the main benchmark.
- Future actual demand is never supplied.
- The forecasting task is a direct 24-step vector refreshed at every hourly origin with a 168-hour historical context.

## 1. Environment and repository paths

Install the repository requirements once in the active Jupyter kernel. The path logic works when the notebook is launched either from the repository root or from the `notebooks/` directory.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

> If packages are not installed yet, uncomment and run the next cell, then restart the kernel if PyTorch or compiled tree libraries were newly installed.

In [ ]:
# %pip install -r ../requirements.txt

## 2. Instantiate the fixed experiment specification

Cluster 1 and Cluster 2 retain the submitted chronological partitions. The two one-year external BDG aggregates use a prespecified 6/3/3-month train/validation/test partition. The percentage ratios do not need to be identical across datasets; the critical requirement is chronological isolation and a sufficiently long development/test period.

In [ ]:
from lash_revision_core import (
    ExperimentConfig, DEFAULT_SPECS, runtime_environment,
    load_base_frame, add_causal_features, build_dataset_bundles,
    split_validation_bundle, search_space_manifest,
)

config = ExperimentConfig(
    data_root=DATA_DIR,
    output_root=OUTPUT_DIR,
    run_profile="paper",
    weather_mode="historical_only",
    save_models=True,
)

display(pd.DataFrame([
    {
        "dataset": spec.name,
        "file": spec.filename,
        "train": f"{spec.train_start} to {pd.Timestamp(spec.train_end)-pd.Timedelta(days=1):%Y-%m-%d}",
        "validation": f"{spec.val_start} to {pd.Timestamp(spec.val_end)-pd.Timedelta(days=1):%Y-%m-%d}",
        "test": f"{spec.test_start} to {pd.Timestamp(spec.test_end)-pd.Timedelta(days=1):%Y-%m-%d}",
        "external_validation": spec.external_validation,
    }
    for spec in config.specs.values()
]))

display(pd.DataFrame([runtime_environment()]))

## 3. Validate the six-column base schema

No model-stage median imputation is allowed. The harmonized files must already be hourly, finite, monotonic, and physically unit-consistent. This turns missing-data handling into an auditable data-preparation decision rather than a hidden model-specific transformation.

In [ ]:
base_summaries = []
for key, spec in config.specs.items():
    df = load_base_frame(spec, DATA_DIR)
    expected = pd.date_range(df.index.min(), df.index.max(), freq="h")
    weekend_ok = bool(((df.index.dayofweek < 5) | (df["Holi"].to_numpy() == 1)).all())
    base_summaries.append({
        "dataset": key,
        "rows": len(df),
        "start": df.index.min(),
        "end": df.index.max(),
        "hourly_continuous": df.index.equals(expected),
        "missing_cells": int(df.isna().sum().sum()),
        "weekends_marked_Holi": weekend_ok,
        "Temp_mean_C": df.Temp.mean(),
        "Humi_mean_pct": df.Humi.mean(),
        "WS_mean_mps": df.WS.mean(),
        "Consumption_mean": df.Consumption.mean(),
    })

base_summary = pd.DataFrame(base_summaries)
display(base_summary)
assert base_summary["hourly_continuous"].all()
assert (base_summary["missing_cells"] == 0).all()
assert base_summary["weekends_marked_Holi"].all()
print("Base-data audit: PASS")

## 4. Rebuild the common causal features

All datasets use the same calendar, thermal, historical-demand, and historical-weather logic. The main no-year-over-year anchor is based only on short-term causal demand context. Annual features are generated only inside the dedicated component-wise sensitivity experiment.

In [ ]:
feature_rows = []
for key, spec in config.specs.items():
    base = load_base_frame(spec, DATA_DIR)
    feat = add_causal_features(base)
    feature_rows.append({
        "dataset": key,
        "rows": len(feat),
        "generated_columns": len(feat.columns),
        "contains_yoy_columns": any("yoy" in c.lower() for c in feat.columns),
        "finite_anchor_after_warmup": bool(np.isfinite(feat["anchor"].iloc[336:]).all()),
    })

display(pd.DataFrame(feature_rows))

## 5. Verify direct 168-to-24 windows and purged validation

The first two-thirds of each validation period are used for hyperparameter tuning. A 23-origin purge prevents overlapping 24-hour target windows from crossing into the held-out calibration segment. Router weights are calibrated only on the final validation segment.

In [ ]:
window_rows = []
for key in config.dataset_keys:
    _, bundles = build_dataset_bundles(config, key)
    tune, cal = split_validation_bundle(bundles["val"])
    for split_name, bundle in [
        ("train", bundles["train"]),
        ("val_tune", tune),
        ("val_calibration", cal),
        ("test", bundles["test"]),
    ]:
        window_rows.append({
            "dataset": key,
            "split": split_name,
            "forecast_origins": len(bundle),
            "past_shape": str(bundle.past.shape),
            "future_shape": str(bundle.future.shape),
            "first_origin": pd.Timestamp(bundle.forecast_origin.min()),
            "last_origin": pd.Timestamp(bundle.forecast_origin.max()),
        })

window_audit = pd.DataFrame(window_rows)
display(window_audit)
print("Window and purge audit: PASS")

## 6. Prespecified model-specific search spaces

An equal number of Optuna trials is not automatically an equal tuning opportunity when search spaces have different dimensionality. The paper profile therefore uses a transparent dimension-adaptive budget, capped between 30 and 60 trials per HPO repeat, and repeats each search with three independent sampler seeds.

The LASH sequential search retains the submitted Appendix-A ranges. The five ensemble-learning families use ranges that contain or extend the settings used in the preceding tree-ensemble studies, while keeping a common chronological validation objective in the current experiment.

In [ ]:
manifest = search_space_manifest()
display(manifest)

budget = pd.DataFrame([
    {
        "model": model,
        "tuned_dimensions": int(dim),
        "trials_per_hpo_repeat": config.hpo_trials(model),
        "hpo_repeats_primary": config.primary_hpo_repeats,
        "hpo_repeats_external": config.external_hpo_repeats,
    }
    for model, dim in __import__("lash_revision_core").MODEL_SEARCH_DIMENSIONS.items()
])
display(budget.sort_values("model"))

### Continuity with the preceding ensemble-learning experiments

The earlier residential ensemble study evaluated the same five tree families and tuned them with GridSearchCV and five-fold cross-validation. Its reported candidate values included RF with 128 trees; GBM with 100/250/500 iterations, learning rates 0.01/0.05/0.1, and depths 5/10; XGBoost with 250/500/1000 iterations, learning rates 0.01/0.05/0.1, depths 6/8/10, and subsampling / feature-sampling values 0.5/0.75/1.0; LightGBM with 1000/1500 iterations and learning rates 0.01/0.05/0.1; and CatBoost with learning rates 0.03/0.1, depths 4/6/10, and L2 regularization levels 1/3/5/7/9.

The present ranges deliberately **contain or extend those earlier settings**, but the validation mechanism is changed. Random or ordinary K-fold cross-validation is not reused here because the current task is a chronology-sensitive hourly rolling 24-step forecast. Every model family is instead tuned on the same chronological validation-tuning segment, separated from router calibration by a 23-origin purge. This difference is methodological harmonization, not an attempt to reproduce the older experiment under a different label.

References used for this continuity check:

- Moon et al., *Sustainable Energy Technologies and Assessments* 54 (2022), 102888.
- Moon et al., *PLOS ONE* 19 (2024), e0307654.
- So et al., *Systems* 11 (2023), 456.


## 7. Export the audit workbook

This workbook is useful as a stable protocol record. It should be generated before the full paper run and kept with the final experiment artifacts.

In [ ]:
audit_path = OUTPUT_DIR / "00_protocol_and_data_audit.xlsx"
with pd.ExcelWriter(audit_path, engine="openpyxl") as writer:
    base_summary.to_excel(writer, sheet_name="Base_Data", index=False)
    window_audit.to_excel(writer, sheet_name="Windows", index=False)
    manifest.to_excel(writer, sheet_name="Search_Spaces", index=False)
    budget.to_excel(writer, sheet_name="HPO_Budgets", index=False)
    pd.DataFrame([runtime_environment()]).to_excel(writer, sheet_name="Environment", index=False)

print("Saved:", audit_path)